In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../beam_search_summary_test.csv")
df.columns

In [ ]:
import pandas as pd

# Melt all sparsity-suffixed columns
id_vars = ['dataset', 'beam_width', 'k']

df_long = pd.wide_to_long(
    df,
    stubnames=['avg_edges', 'train_relevant', 'train_seen', 'train_expanded'],
    i=id_vars,
    j='coverage',
    sep='_',
    suffix=r'[\d.]+'
).reset_index()

df_long['coverage'] = df_long['coverage'].astype(float)
df_long

In [ ]:
df_long['recall'] = df_long['train_relevant'] / df_long['k']
baseline = df_long[df_long['coverage'] == 1.0].copy()
novel    = df_long[df_long['coverage'] <  1.0].copy()
baseline_ref = baseline[['dataset', 'beam_width', 'k', 'recall', 'train_seen', 'avg_edges']] \
    .rename(columns={'recall': 'baseline_recall', 'train_seen': 'baseline_seen', 'avg_edges': 'baseline_avg_edges'})

novel = novel.merge(baseline_ref, on=['dataset', 'beam_width', 'k'])
# novel['recall_ratio'] = novel['recall'] / novel['baseline_recall']
# novel['recall_diff'] = novel['recall'] - novel['baseline_recall']
# novel['compute_ratio'] = novel['train_seen'] / novel['baseline_seen']
# novel['degree_ratio'] = novel['avg_edges'] / novel['baseline_avg_edges']

In [ ]:
# 1. Pick the single best baseline: coverage=1.0, highest beam_width
best_baseline = (
    df_long[df_long['coverage'] == 1.0]
    .sort_values('beam_width', ascending=False)
    .groupby(['dataset', 'k'], as_index=False)
    .first()
)[['dataset', 'k', 'recall', 'train_seen', 'avg_edges']] \
 .rename(columns={
     'recall':     'baseline_recall',
     'train_seen': 'baseline_seen',
     'avg_edges':  'baseline_avg_edges'
 })

# 2. Merge on (dataset, k) only — no beam_width
novel = df_long[df_long['coverage'] < 1.0].copy()
novel = novel.merge(best_baseline, on=['dataset', 'k'])

# novel['recall_ratio']  = novel['recall']      / novel['baseline_recall']
# novel['recall_diff']   = novel['recall']      - novel['baseline_recall']
# novel['compute_ratio'] = novel['train_seen']  / novel['baseline_seen']
# novel['degree_ratio']  = novel['avg_edges']   / novel['baseline_avg_edges']

In [ ]:
novel.columns, baseline.columns

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def pareto_frontier(df, x_col, y_col):
    pts = df[[x_col, y_col]].values
    is_pareto = np.ones(len(pts), dtype=bool)
    for i in range(len(pts)):
        if not is_pareto[i]:
            continue
        dominated = (pts[:, 0] <= pts[i, 0]) & (pts[:, 1] >= pts[i, 1])
        dominated[i] = False
        if dominated.any():
            is_pareto[i] = False
    return df[is_pareto].sort_values(x_col)

datasets = sorted(novel['dataset'].unique())
ks       = sorted(novel['k'].unique())

fig = make_subplots(
    rows=len(datasets), cols=len(ks),
    subplot_titles=[f'{d}  |  Recall@{k}' for d in datasets for k in ks],
    horizontal_spacing=0.06,
    vertical_spacing=0.12,
)

for row_idx, dataset in enumerate(datasets, start=1):
    baseline_bw = int(
        df_long[(df_long['dataset'] == dataset) & (df_long['coverage'] == 1.0)]['beam_width'].max()
    )

    for col_idx, k in enumerate(ks, start=1):
        sub = novel[(novel['dataset'] == dataset) & (novel['k'] == k)].copy()
        if sub.empty:
            continue

        if 'recall_diff' not in sub.columns:
            sub['recall_diff'] = sub['recall'] - sub['baseline_recall']

        ref             = sub.iloc[0]
        baseline_recall = ref['baseline_recall']
        baseline_seen   = ref['baseline_seen']

        full_cov = df_long[
            (df_long['dataset'] == dataset) & (df_long['k'] == k) & (df_long['coverage'] == 1.0)
        ].copy().sort_values('beam_width')

        show_legend = (row_idx == 1 and col_idx == 1)

        # ── Full-coverage reference curve ────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=full_cov['train_seen'],
            y=full_cov['recall'],
            mode='lines+markers',
            name='Full coverage',
            legendgroup='full_cov',
            showlegend=show_legend,
            line=dict(color='crimson', width=2, dash='dot'),
            marker=dict(symbol='diamond', size=9, color='crimson',
                        line=dict(color='black', width=1)),
            customdata=full_cov['beam_width'].values[:, None],
            hovertemplate=(
                '<b>Full Coverage</b><br>'
                'beam_width: %{customdata[0]:.0f}<br>'
                'recall: %{y:.4f}<br>'
                'train_seen: %{x:.0f}'
                '<extra></extra>'
            ),
        ), row=row_idx, col=col_idx)

        # ── Novel scatter (all grey) ─────────────────────────────────────────
        fig.add_trace(go.Scatter(
            x=sub['train_seen'],
            y=sub['recall'],
            mode='markers',
            name='Novel',
            legendgroup='novel',
            showlegend=show_legend,
            marker=dict(color='#aaaaaa', size=6, opacity=0.4),
            customdata=np.column_stack([
                sub['beam_width'].values,
                sub['coverage'].values,
                sub['recall_diff'].values,
                sub['avg_edges'].values,
            ]),
            hovertemplate=(
                '<b>Novel</b><br>'
                'beam_width: %{customdata[0]:.0f}<br>'
                'coverage: %{customdata[1]:.3f}<br>'
                'recall: %{y:.4f}<br>'
                'recall_diff: %{customdata[2]:+.4f}<br>'
                'train_seen: %{x:.0f}<br>'
                'avg_edges: %{customdata[3]:.2f}'
                '<extra></extra>'
            ),
        ), row=row_idx, col=col_idx)

        # ── Pareto frontier ──────────────────────────────────────────────────
        frontier = pareto_frontier(
            sub[['train_seen', 'recall', 'beam_width', 'coverage', 'recall_diff']].dropna(),
            'train_seen', 'recall'
        )
        fig.add_trace(go.Scatter(
            x=frontier['train_seen'],
            y=frontier['recall'],
            mode='lines+markers',
            name='Pareto frontier',
            legendgroup='pareto',
            showlegend=show_legend,
            line=dict(color='black', width=2, dash='dash'),
            marker=dict(color='black', size=10, symbol='circle',
                        line=dict(color='white', width=1.5)),
            customdata=np.column_stack([
                frontier['beam_width'].values,
                frontier['coverage'].values,
                frontier['recall_diff'].values,
            ]),
            hovertemplate=(
                '<b>Pareto Point</b><br>'
                'beam_width: %{customdata[0]:.0f}<br>'
                'coverage: %{customdata[1]:.3f}<br>'
                'recall: %{y:.4f}<br>'
                'recall_diff: %{customdata[2]:+.4f}<br>'
                'train_seen: %{x:.0f}<br>'
                'avg_edges: %{customdata[3]:.2f}'
                '<extra></extra>'
            ),
        ), row=row_idx, col=col_idx)

        fig.add_hline(y=baseline_recall, line=dict(color='crimson', width=1, dash='dot'), row=row_idx, col=col_idx)
        fig.add_vline(x=baseline_seen,   line=dict(color='crimson', width=1, dash='dot'), row=row_idx, col=col_idx)

fig.update_layout(
    title=dict(text=f'<b>Recall vs Distance Computations</b>   <span style="font-size:12px;color:gray">Baseline: bw={baseline_bw}, coverage=1.0</span>'),
    height=400 * len(datasets),
    hovermode='closest',
    legend=dict(bgcolor='rgba(255,255,255,0.85)', bordercolor='lightgray', borderwidth=1),
)
fig.update_xaxes(title_text='Distance Computations (train_seen)')
fig.update_yaxes(title_text='Recall')

fig.write_html('/Users/pratyushavi/Developer/WebDev/scoogleystinkbomb/notes/navigable-graphs/recall_plots.html')
fig.show()


In [ ]:
def top_k_bw_coverage(df_novel, top_k=5, group_by=('dataset', 'k'), score_col='recall'):
    """
    Return the top-k (beam_width, coverage) combinations per group, ranked by score_col descending.
    """
    return (
        df_novel
        .sort_values(score_col, ascending=False)
        .groupby(list(group_by), sort=False)
        .head(top_k)
        [list(group_by) + ['beam_width', 'coverage', score_col]]
        .reset_index(drop=True)
    )

def pareto_bw_coverage(df_novel, xcol='train_seen', score_col='recall', group_by=('dataset', 'k')):
    """
    Return Pareto-optimal (beam_width, coverage) combinations per group.
    A row is Pareto-optimal if no other row in the group has both higher score_col and lower xcol.
    """
    pieces = [
        pareto_frontier(g, xcol, score_col)
        for _, g in df_novel.groupby(list(group_by), sort=False)
    ]
    result = pd.concat(pieces, ignore_index=True)
    return result[list(group_by) + ['beam_width', 'coverage', score_col, xcol]]

def plot_top_k_settings(df_novel, df_baseline, score_col='recall', save=False):
    """
    For each dataset: one figure with rows=recall@k values, cols=[distance comps, avg out-edges].
    Pareto-optimal novel (bw, coverage) combos shown with distinct marker+color; baseline as a dashed line.
    """
    datasets = sorted(df_novel['dataset'].unique())
    k_values = sorted(df_novel['k'].unique())
    n_k      = len(k_values)

    for dataset in datasets:
        nov_ds  = df_novel[df_novel['dataset'] == dataset]
        base_ds = df_baseline[df_baseline['dataset'] == dataset]

        # Union of Pareto-optimal combos across all k values for this dataset
        pareto_per_k = pareto_bw_coverage(nov_ds, group_by=('k',), score_col=score_col)
        unique_combos = (
            pareto_per_k[['beam_width', 'coverage']]
            .drop_duplicates()
            .sort_values(['beam_width', 'coverage'])
            .reset_index(drop=True)
        )

        fig, axes = plt.subplots(n_k, 2, figsize=(14, 5 * n_k), squeeze=False)
        fig.suptitle(f'Dataset: {dataset}', fontsize=14, fontweight='bold')

        for row_idx, k in enumerate(k_values):
            nov_k  = nov_ds[nov_ds['k'] == k]
            base_k = base_ds[base_ds['k'] == k]

            ax_dist  = axes[row_idx, 0]
            ax_edges = axes[row_idx, 1]

            for ax, xcol, xlabel in [
                (ax_dist,  'train_seen', 'Distance Computations'),
                (ax_edges, 'avg_edges',  'Avg. Out-Edges'),
            ]:
                # Baseline: line connecting bw values at coverage=1.0
                base_sorted = base_k.sort_values(xcol)
                ax.plot(
                    base_sorted[xcol], base_sorted['recall'],
                    color='black', linestyle='--', linewidth=1.5,
                    marker='x', markersize=7, zorder=3, label='Baseline (cov=1.0, bw varies)',
                )

                # Pareto-optimal combos
                for i, (_, combo) in enumerate(unique_combos.iterrows()):
                    bw, cov = combo['beam_width'], combo['coverage']
                    row = nov_k[(nov_k['beam_width'] == bw) & (nov_k['coverage'] == cov)]
                    if row.empty:
                        continue
                    ax.scatter(
                        row[xcol].values[0], row['recall'].values[0],
                        marker=_MARKERS[i % len(_MARKERS)],
                        color=_COLORS[i % len(_COLORS)],
                        s=90, zorder=5, label=f'bw={int(bw)}, cov={cov:.2f}',
                    )

                ax.set_xlabel(xlabel, fontsize=10)
                ax.set_ylabel(f'Recall@{k}', fontsize=10)
                ax.set_title(f'Recall@{k} vs {xlabel}', fontsize=11)
                ax.legend(
                    loc='upper center', bbox_to_anchor=(0.5, -0.22),
                    ncol=3, fontsize=8, frameon=True,
                )

        plt.tight_layout()
        if save:
            fig.savefig(f'{dataset}_pareto.png', bbox_inches='tight', dpi=150)
        plt.show()

In [ ]:
def pareto_frontier(df, x_col, y_col):
    pts = df[[x_col, y_col]].values
    is_pareto = np.ones(len(pts), dtype=bool)
    for i in range(len(pts)):
        if not is_pareto[i]:
            continue
        dominated = (pts[:, 0] <= pts[i, 0]) & (pts[:, 1] >= pts[i, 1])
        dominated[i] = False
        if dominated.any():
            is_pareto[i] = False
    return df[is_pareto].sort_values(x_col)

def plot_recall_vs_seen(df_novel, df_baseline, save=False):
    """
    One figure per dataset, with one subplot per k value.
    Recall@k vs train_seen: baseline in red (line through points), novel points in grey,
    Pareto-optimal novel points in black, a star on the highest-recall novel point
    that lies above the baseline curve.
    """
    datasets = sorted(df_novel['dataset'].unique())
    k_values = sorted(df_novel['k'].unique(), reverse=True)
    n_k      = len(k_values)

    for dataset in datasets:
        nov_ds  = df_novel[df_novel['dataset'] == dataset]
        base_ds = df_baseline[df_baseline['dataset'] == dataset]

        fig, axes = plt.subplots(1, n_k, figsize=(6 * n_k, 5), squeeze=False)
        fig.suptitle(f'Dataset: {dataset}', fontsize=14, fontweight='bold')

        for col_idx, k in enumerate(k_values):
            ax     = axes[0, col_idx]
            nov_k  = nov_ds[nov_ds['k'] == k]
            base_k = base_ds[base_ds['k'] == k].sort_values('train_seen')

            # Novel points in grey
            ax.scatter(
                nov_k['train_seen'], nov_k['recall'],
                color='grey', s=20, alpha=0.5, zorder=2, label='Novel',
            )

            # Pareto-optimal novel points in black
            frontier = pareto_frontier(
                nov_k[['train_seen', 'recall']].dropna(), 'train_seen', 'recall'
            )
            ax.plot(
                frontier['train_seen'], frontier['recall'],
                color='black', marker='o', markersize=6, linewidth=1.5,
                zorder=4, label='Pareto frontier',
            )

            # Baseline in red with a line through the points
            ax.plot(
                base_k['train_seen'], base_k['recall'],
                color='red', marker='o', markersize=6, linewidth=1.5,
                zorder=3, label='Baseline (cov=1.0)',
            )

            # Star on the highest-recall novel point lying above the baseline curve.
            # All baseline runs share the same graph (cov=1.0), so avg_edges is constant.
            if not nov_k.empty and not base_k.empty:
                base_edges = base_k['avg_edges'].iloc[0]
                # Interpolate baseline recall at each novel point's train_seen
                base_recall_at = np.interp(
                    nov_k['train_seen'], base_k['train_seen'], base_k['recall']
                )
                above = nov_k[nov_k['recall'].values > base_recall_at]
                if not above.empty:
                    best = above.sort_values(
                        ['recall', 'train_seen'], ascending=[False, True]
                    ).iloc[0]
                    ax.scatter(
                        best['train_seen'], best['recall'],
                        marker='*', s=350, color='gold', edgecolor='black',
                        linewidth=1.0, zorder=6,
                    )

                    pct_fewer = (1 - best['avg_edges'] / base_edges) * 100
                    blurb = (
                        f"Coverage = {best['coverage']:.3f}\n"
                        f"Recall = {best['recall']:.4f}\n"
                        f"Avg. Degree = {best['avg_edges']:.1f}"
                    )
                    ax.annotate(
                        blurb,
                        xy=(best['train_seen'], best['recall']),
                        xytext=(0.8, 0.5), textcoords='axes fraction',
                        ha='right', va='bottom', fontsize=12,
                        bbox=dict(fc='white',
                                  ec='black', alpha=0.9),
                        arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
                    )

            ax.set_xlabel('Distance Computations', fontsize=10)
            ax.set_ylabel(f'Recall@{k}', fontsize=10)
            ax.set_title(f'Recall@{k} vs Distance Computations', fontsize=11)
            ax.legend(fontsize=9)

        plt.tight_layout()
        if save:
            fig.savefig(f'{dataset}_recall_vs_seen.png', bbox_inches='tight', dpi=150)
        plt.show()

plot_recall_vs_seen(novel, baseline)

In [ ]:
import matplotlib.ticker as mticker

def _k_formatter(x, pos):
    """Format axis ticks in thousands with a 'k' suffix: 500 -> 0.5k, 1000 -> 1k, 12500 -> 12.5k."""
    if x == 0:
        return '0'
    v = x / 1000.0
    # Drop trailing .0 (1.0k -> 1k) but keep fractional thousands (0.5k, 12.5k)
    s = f'{v:.1f}'.rstrip('0').rstrip('.')
    return f'{s}k'

def plot_recall_vs_seen_paper(df_novel, df_baseline, width_in=14.0, save_path=None):
    """
    Compact, paper-ready figure: single figure, rows = k values, cols = datasets.
    - Pareto frontier black, baseline red.
    - star = highest-recall novel point above the baseline curve (marker only, no on-plot text).
    - one shared legend and one shared x-axis label; x-axis shared per column (each column is one dataset).
    - x ticks in thousands with a 'k' suffix (e.g. 0.5k, 1k, 12.5k): exactly 3 per axis, first tick at 0.
    Returns a DataFrame of the starred operating points (for the caption / a table).
    """
    datasets = sorted(df_novel['dataset'].unique())
    k_values = sorted(df_novel['k'].unique(), reverse=True)
    n_ds, n_k = len(datasets), len(k_values)

    # Paper-scale typography (relative to the small final size)
    plt.rcParams.update({
        'font.size': 7, 'axes.titlesize': 8, 'axes.labelsize': 8,
        'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 7,
        'lines.linewidth': 1.0, 'lines.markersize': 3,
        'figure.dpi': 200,            # crisper inline preview
        'savefig.dpi': 600,           # crisper raster export (PNG); PDF stays vector
    })

    cell_w = width_in / n_ds                      # ~equal cell widths
    fig, axes = plt.subplots(
        n_k, n_ds, figsize=(width_in, cell_w * 0.85 * n_k),
        sharex='col', squeeze=False,
    )

    star_rows = []
    handles_labels = None

    for c, dataset in enumerate(datasets):
        nov_ds  = df_novel[df_novel['dataset'] == dataset]
        base_ds = df_baseline[df_baseline['dataset'] == dataset]

        # Shared x-range for this dataset column: 0 .. max train_seen seen anywhere in the column
        col_xmax = max(
            float(nov_ds['train_seen'].max() if not nov_ds.empty else 0.0),
            float(base_ds['train_seen'].max() if not base_ds.empty else 0.0),
        )

        for r, k in enumerate(k_values):
            ax     = axes[r, c]
            nov_k  = nov_ds[nov_ds['k'] == k]
            base_k = base_ds[base_ds['k'] == k].sort_values('train_seen')

            # ax.scatter(nov_k['train_seen'], nov_k['recall'],
            #            color='grey', s=8, alpha=0.5, zorder=2, label='Novel')

            frontier = pareto_frontier(
                nov_k[['train_seen', 'recall']].dropna(), 'train_seen', 'recall')
            ax.plot(frontier['train_seen'], frontier['recall'],
                    color='black', marker='o', zorder=3, alpha=0.8, label='almost navigable graphs')

            ax.plot(base_k['train_seen'], base_k['recall'],
                    color='red', marker='o', zorder=4, alpha=1, label='fully navigable graphs')

            # Star: highest-recall novel point above the baseline curve.
            # All baseline runs share the same graph (cov=1.0), so avg_edges is constant.
            if not nov_k.empty and not base_k.empty:
                base_edges = base_k['avg_edges'].iloc[0]
                base_recall_at = np.interp(
                    nov_k['train_seen'], base_k['train_seen'], base_k['recall'])
                above = nov_k[nov_k['recall'].values > base_recall_at]
                if not above.empty:
                    best = above.sort_values(
                        ['recall', 'train_seen'], ascending=[False, True]).iloc[0]
                    ax.scatter(best['train_seen'], best['recall'],
                               marker='*', s=100, color='gold', edgecolor='black',
                               linewidth=0.6, zorder=6, label='best almost navigable configuration')
                    star_rows.append({
                        'dataset': dataset, 'k': k,
                        'beam_width': int(best['beam_width']),
                        'coverage': round(float(best['coverage']), 3),
                        'recall': round(float(best['recall']), 4),
                        'avg_edges': round(float(best['avg_edges']), 2),
                        'pct_fewer_edges': round((1 - best['avg_edges'] / base_edges) * 100, 1),
                    })

            # Exactly 3 x ticks at [0, mid, max], in thousands with a 'k' suffix (0, 0.5k, 1k)
            if col_xmax > 0:
                ax.set_xlim(0, col_xmax)
                ax.xaxis.set_major_locator(
                    mticker.FixedLocator([0, col_xmax / 2, col_xmax]))
            ax.xaxis.set_major_formatter(mticker.FuncFormatter(_k_formatter))

            if r == 0:
                ax.set_title(dataset)
            if c == 0:
                ax.set_ylabel(f'Recall@{k}')
            ax.tick_params(length=2, pad=1)

            if handles_labels is None and ax.get_legend_handles_labels()[0]:
                handles_labels = ax.get_legend_handles_labels()

    # Single shared x-axis label, lifted just under the bottom row of plots
    fig.supxlabel('Distance Computations', fontsize=8, y=0.08)

    if handles_labels:
        fig.legend(*handles_labels, loc='lower center', ncol=4,
                   bbox_to_anchor=(0.5, -0.03), frameon=False)

    # Leave a small band at the bottom for the lifted label + legend
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    plt.show()

    return pd.DataFrame(star_rows)

star_table = plot_recall_vs_seen_paper(
    novel, baseline, width_in=7.0, save_path='recall_vs_seen_paper.pdf')
star_table

In [ ]:
def plot_degree_vs_coverage_paper(df_long, width_in=7.0, save_path=None):
    """
    Compact, paper-ready figure: one subplot per dataset (single row).
    Avg. out-degree (y) vs coverage (x). avg_edges depends only on (dataset, coverage),
    so it collapses to one curve per dataset (independent of beam_width / k).
    x-axis ticks snapped to nearest 0.01; y-axis spans [0, ceil(max degree)] with 5 ticks;
    one shared x-label. x-limits have slack so endpoint markers aren't clipped.
    """
    datasets = sorted(df_long['dataset'].unique())
    n_ds = len(datasets)

    plt.rcParams.update({
        'font.size': 7, 'axes.titlesize': 8, 'axes.labelsize': 8,
        'xtick.labelsize': 4.5, 'ytick.labelsize': 4.5, 'legend.fontsize': 7,
        'lines.linewidth': 1.0, 'lines.markersize': 2,
        'figure.dpi': 200, 'savefig.dpi': 600,
    })

    cell_w = width_in / n_ds
    fig, axes = plt.subplots(1, n_ds, figsize=(width_in, cell_w * 0.85), squeeze=False)

    for c, dataset in enumerate(datasets):
        ax = axes[0, c]
        # One avg_edges per (dataset, coverage) — collapse duplicates across bw / k
        curve = (
            df_long[df_long['dataset'] == dataset]
            .groupby('coverage', as_index=False)['avg_edges']
            .first()
            .sort_values('coverage')
        )

        ax.plot(curve['coverage'], curve['avg_edges'],
                color='black', marker='o', zorder=3)

        # x: 5 ticks snapped to the nearest 0.01, within [cmin, cmax] + slack on the limits
        cmin, cmax = curve['coverage'].min(), curve['coverage'].max()
        xticks = np.unique(np.round(np.linspace(cmin, cmax, 5), 2))
        xpad = 0.03 * (cmax - cmin)
        ax.set_xlim(cmin - xpad, cmax + xpad)
        ax.xaxis.set_major_locator(mticker.FixedLocator(xticks))
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x:g}'))

        # y: 5 ticks spanning [0, ceil(max degree)]
        ytop = np.ceil(curve['avg_edges'].max())
        yticks = np.round(np.linspace(0, ytop, 5))
        ax.set_ylim(0, ytop)
        ax.yaxis.set_major_locator(mticker.FixedLocator(yticks))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, p: f'{y:g}'))

        ax.set_title(dataset)
        if c == 0:
            ax.set_ylabel('Avg. Out-Degree')
        ax.tick_params(length=2, pad=1)

    # Single shared x-axis label
    fig.supxlabel(r'Coverage $= (1 - \gamma)$', fontsize=8)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    plt.show()

plot_degree_vs_coverage_paper(df_long, width_in=7.0, save_path='degree_vs_coverage_paper.pdf')

In [ ]:
def build_comparison_table(df_long, min_recall=0.9):
    """
    For each (dataset, k): two rows.
      - 'best novel' = highest-recall point above the baseline curve (the star in the plots).
      - 'baseline'   = the cheapest cov=1.0 setting (min train_seen) whose recall both beats the
                       novel best AND is >= min_recall, i.e. recall >= max(best_recall, min_recall).
                       Falls back to the highest-recall baseline if none qualify.
    Columns: Dataset, Recall@k, Coverage, Distance Computations, Recall, Avg. Degree, Beam Width.
    """
    datasets = sorted(df_long['dataset'].unique())
    k_values = sorted(df_long['k'].unique())

    rows = []
    for dataset in datasets:
        ds = df_long[df_long['dataset'] == dataset]
        for k in k_values:
            dk = ds[ds['k'] == k]
            base_k = dk[dk['coverage'] == 1.0].sort_values('train_seen')
            nov_k  = dk[dk['coverage'] < 1.0]
            if base_k.empty or nov_k.empty:
                continue

            # --- best novel point: highest recall above the interpolated baseline curve ---
            base_recall_at = np.interp(
                nov_k['train_seen'], base_k['train_seen'], base_k['recall'])
            above = nov_k[nov_k['recall'].values > base_recall_at]
            if above.empty:
                continue
            best = above.sort_values(
                ['recall', 'train_seen'], ascending=[False, True]).iloc[0]

            # --- baseline: cheapest cov=1.0 setting beating the novel best in recall while
            #     also reaching at least min_recall, i.e. recall >= max(best, min_recall).
            #     Falls back to the highest-recall baseline if none qualify. ---
            target_recall = max(best['recall'], min_recall)
            beats = base_k[base_k['recall'] >= target_recall]
            base_match = (beats.sort_values('train_seen').iloc[0] if not beats.empty
                          else base_k.sort_values('recall').iloc[-1])

            label = f'Recall@{k}'
            # baseline row first, then the best-novel row
            rows.append({
                'Dataset': dataset, 'Recall@k': label, 'Coverage': base_match['coverage'],
                'Distance Computations': base_match['train_seen'], 'Recall': base_match['recall'],
                'Avg. Degree': base_match['avg_edges'], 'Beam Width': int(base_match['beam_width']),
            })
            rows.append({
                'Dataset': dataset, 'Recall@k': label, 'Coverage': best['coverage'],
                'Distance Computations': best['train_seen'], 'Recall': best['recall'],
                'Avg. Degree': best['avg_edges'], 'Beam Width': int(best['beam_width']),
            })

    cols = ['Dataset', 'Recall@k', 'Coverage', 'Distance Computations',
            'Recall', 'Avg. Degree', 'Beam Width']
    return pd.DataFrame(rows, columns=cols)

comparison_table = build_comparison_table(df_long)
comparison_table.round({'Coverage': 3, 'Distance Computations': 1, 'Recall': 4, 'Avg. Degree': 2})

In [ ]:
def comparison_table_to_latex(df, caption=None, label=None):
    """
    Render the comparison table fully gridded (vertical rule between every column) and
    spanning both columns (table*), with multirow label cells.
      - Dataset spans its 6 rows, Recall@k spans its 2 rows.
      - Each k-group: baseline row first, best-novel row second.
      - Within each pair the better value is bolded per column:
        lower Distance Computations, higher Recall, lower Avg. Degree.
      - \\cline{2-7} between k-groups (keeps the Dataset multirow cell open),
        full \\hline between datasets, and outer borders.
    Requires \\usepackage{multirow} in the preamble (uses \\hline/\\cline, not booktabs).
    """
    # vertical rule between every column, plus outer borders
    col_fmt = '|l|l|' + '|'.join('r' * 5) + '|'   # |l|l|r|r|r|r|r|
    headers = ['Dataset', 'Recall@$k$', 'Coverage', 'Dist.\\ Comp.',
               'Recall', 'Avg.\\ Deg.', 'Beam Width']

    def tex(s):
        # escape underscores in text-mode labels (dataset names like coco_i2i)
        return str(s).replace('_', r'\_')

    def cell(val, fmt, bold):
        s = format(val, fmt)
        return rf'\textbf{{{s}}}' if bold else s

    def render_pair(base, nov):
        """Return (base_cells, nov_cells) with the per-column winner bolded."""
        # winner = lower dist, higher recall, lower degree (compared between the two rows)
        dc_base_wins = base['Distance Computations'] < nov['Distance Computations']
        rc_base_wins = base['Recall'] > nov['Recall']
        ad_base_wins = base['Avg. Degree'] < nov['Avg. Degree']

        def row(r, is_base):
            return [
                cell(r['Coverage'], '.3f', False),
                cell(r['Distance Computations'], '.0f', dc_base_wins == is_base),
                cell(r['Recall'], '.4f', rc_base_wins == is_base),
                cell(r['Avg. Degree'], '.2f', ad_base_wins == is_base),
                cell(int(r['Beam Width']), 'd', False),
            ]
        return row(base, True), row(nov, False)

    lines = [
        r'\begin{table*}[t]',
        r'\centering',
        r'\small',
        r'\setlength{\tabcolsep}{4pt}',
        rf'\begin{{tabular}}{{{col_fmt}}}',
        r'\hline',
        ' & '.join(headers) + r' \\',
        r'\hline',
    ]

    datasets = list(dict.fromkeys(df['Dataset']))
    for di, dataset in enumerate(datasets):
        ds = df[df['Dataset'] == dataset]
        ks = list(dict.fromkeys(ds['Recall@k']))
        ds_cell = rf'\multirow{{{len(ds)}}}{{*}}{{{tex(dataset)}}}'

        first_ds_row = True
        for ki, klabel in enumerate(ks):
            grp = ds[ds['Recall@k'] == klabel]          # [baseline, novel]
            base, nov = grp.iloc[0], grp.iloc[1]
            base_cells, nov_cells = render_pair(base, nov)
            k_cell = rf'\multirow{{2}}{{*}}{{{tex(klabel)}}}'

            lead_ds = ds_cell if first_ds_row else ''
            lines.append(' & '.join([lead_ds, k_cell] + base_cells) + r' \\')
            lines.append(' & '.join(['', ''] + nov_cells) + r' \\')
            first_ds_row = False

            # rule between k-groups; \cline{2-7} keeps the Dataset multirow cell open
            if ki < len(ks) - 1:
                lines.append(r'\cline{2-7}')
        # full-width rule between datasets
        lines.append(r'\hline')

    lines += [r'\end{tabular}']
    if caption:
        lines.append(rf'\caption{{{caption}}}')
    if label:
        lines.append(rf'\label{{{label}}}')
    lines.append(r'\end{table*}')
    return '\n'.join(lines)


latex = comparison_table_to_latex(
    comparison_table,
    caption=(r'Best almost-navigable configuration vs.\ the cheapest fully-navigable '
             r'baseline reaching comparable recall, per dataset and $k$. Within each pair '
             r'the better value is bolded (lower distance computations, higher recall, '
             r'lower average degree).'),
    label='tab:almost-navigable',
)
print(latex)

In [ ]:
from IPython.display import HTML

def comparison_table_to_html(df):
    """
    Preview the comparison table inline with merged Dataset / Recall@k cells (rowspan),
    bold novel rows, and booktabs-style horizontal rules. No LaTeX install needed.
    """
    headers = ['Dataset', 'Recall@<i>k</i>', 'Coverage', 'Dist. Comp.',
               'Recall', 'Avg. Deg.', 'Beam Width']

    css = """
    <style>
    table.cmp { border-collapse: collapse; font-family: serif; font-size: 13px; }
    table.cmp th, table.cmp td { padding: 3px 10px; text-align: right; }
    table.cmp th { border-top: 2px solid #000; border-bottom: 1.5px solid #000; }
    table.cmp td.lbl { text-align: left; vertical-align: middle; border-right: 1px solid #ddd; }
    table.cmp tr.dsstart td { border-top: 1.5px solid #000; }   /* midrule between datasets */
    table.cmp tr.kstart td.num { border-top: 1px solid #bbb; }  /* cmidrule between k-groups */
    table.cmp tr.last td { border-bottom: 2px solid #000; }
    table.cmp tr.novel td.num { font-weight: bold; }
    </style>
    """

    def numcells(r):
        return [
            f"{r['Coverage']:.3f}",
            f"{r['Distance Computations']:.0f}",
            f"{r['Recall']:.4f}",
            f"{r['Avg. Degree']:.2f}",
            f"{int(r['Beam Width'])}",
        ]

    body, datasets = [], list(dict.fromkeys(df['Dataset']))
    total, seen = len(df), 0
    for dataset in datasets:
        ds = df[df['Dataset'] == dataset]
        ks = list(dict.fromkeys(ds['Recall@k']))
        ds_first = True
        for klabel in ks:
            grp = ds[ds['Recall@k'] == klabel]   # [baseline, novel]
            for ri, (_, r) in enumerate(grp.iterrows()):
                seen += 1
                classes = []
                if ds_first and ri == 0:
                    classes.append('dsstart')
                if ri == 0:
                    classes.append('kstart')
                if ri == 1:
                    classes.append('novel')
                if seen == total:
                    classes.append('last')
                tr = f'<tr class="{" ".join(classes)}">'

                cells = ''
                if ds_first and ri == 0:
                    cells += f'<td class="lbl" rowspan="{len(ds)}">{dataset}</td>'
                if ri == 0:
                    cells += f'<td class="lbl" rowspan="2">{klabel}</td>'
                cells += ''.join(f'<td class="num">{c}</td>' for c in numcells(r))
                body.append(tr + cells + '</tr>')
                ds_first = False

    head = '<tr>' + ''.join(f'<th>{h}</th>' for h in headers) + '</tr>'
    return HTML(css + '<table class="cmp">' + head + ''.join(body) + '</table>')

comparison_table_to_html(comparison_table)

In [11]:
def coverage_stats_to_latex(csv_path='../coverage_stats.csv',
                            coverages=(100.0, 99.5), method='robust-prune',
                            caption=None, label=None):
    """
    LaTeX table from coverage_stats.csv: <method> runs at the given coverage levels.
    Drops the method/metric columns, groups the degree statistics under
    Out-degree (Mean/Median/Min/Max) and In-degree (Median/Min/Max) super-columns, and merges
    each dataset's Dataset / Points / Dim. cells across its coverage rows with \\multirow
    (these are constant per dataset). The leading label headers are vertically centred too.
    Requires \\usepackage{booktabs} and \\usepackage{multirow}.
    Note: the CSV has no 'mean in degree' column, so In-degree has no Mean sub-column.
    In-degree prints '-' when only a partial point set was computed (points computed < total),
    since the in-degree counts are unreliable when most of the graph was not traversed.
    """
    stats = pd.read_csv(csv_path)
    f = (stats[(stats['method'] == method) & (stats['coverage'].isin(coverages))]
         .drop(columns=['method', 'metric'])
         .sort_values(['dataset', 'coverage'], ascending=[True, False])
         .reset_index(drop=True))

    # centred numeric columns: dataset | points dim cov | out(4) | in(3)
    col_fmt = 'l' + 'ccc' + 'cccc' + 'ccc'

    def tex(s):
        return str(s).replace('_', r'\_')

    def in_unavailable(r):
        # in-degree is unreliable when only part of the point set was computed
        return r['points computed'] < r['total points']

    def fmt_row(r, span=None):
        # Points and Dim. are constant per dataset: \multirow on the first row, blank after.
        if span is not None:
            pts = rf"\multirow{{{span}}}{{*}}{{{int(r['total points']):,}}}"
            dim = rf"\multirow{{{span}}}{{*}}{{{int(r['dimensions'])}}}"
        else:
            pts, dim = '', ''
        cells = [
            pts,
            dim,
            f"{r['coverage']:.1f}",
            # out-degree: mean, median, min, max
            f"{r['mean out degree']:.2f}",
            f"{r['median out degree']:.1f}",
            f"{int(r['min out degree'])}",
            f"{int(r['max out degree'])}",
        ]
        # in-degree: median, min, max (no mean in CSV)
        if in_unavailable(r):
            cells += ['-', '-', '-']
        else:
            cells += [
                f"{r['median in degree']:.1f}",
                f"{int(r['min in degree'])}",
                f"{int(r['max in degree'])}",
            ]
        return cells

    # leading label headers span both header rows (vertically centred)
    h_dataset = r'\multirow{2}{*}{Dataset}'
    h_points  = r'\multirow{2}{*}{Points}'
    h_dim     = r'\multirow{2}{*}{Dim.}'
    h_cov     = r'\multirow{2}{*}{Coverage\ (\%)}'

    lines = [
        r'\begin{table*}[t]', r'\centering', r'\small',
        r'\setlength{\tabcolsep}{4pt}',
        rf'\begin{{tabular}}{{{col_fmt}}}',
        r'\toprule',
        rf'{h_dataset} & {h_points} & {h_dim} & {h_cov} & '
        r'\multicolumn{4}{c}{Out-degree} & \multicolumn{3}{c}{In-degree} \\',
        r'\cmidrule(lr){5-8} \cmidrule(lr){9-11}',
        r'& & & & Mean & Median & Min & Max & Median & Min & Max \\',
        r'\midrule',
    ]

    datasets = list(dict.fromkeys(f['dataset']))
    for di, ds in enumerate(datasets):
        grp = f[f['dataset'] == ds]
        n = len(grp)
        ds_cell = rf'\multirow{{{n}}}{{*}}{{{tex(ds)}}}'
        for ri, (_, r) in enumerate(grp.iterrows()):
            lead = ds_cell if ri == 0 else ''
            cells = fmt_row(r, span=n if ri == 0 else None)
            lines.append(' & '.join([lead] + cells) + r' \\')
        if di < len(datasets) - 1:
            lines.append(r'\cmidrule(l){2-11}')

    lines += [r'\bottomrule', r'\end{tabular}']
    if caption:
        lines.append(rf'\caption{{{caption}}}')
    if label:
        lines.append(rf'\label{{{label}}}')
    lines.append(r'\end{table*}')
    return '\n'.join(lines)


print(coverage_stats_to_latex(
    caption=(r'Graph degree statistics for robust-prune runs at $100\%$ and '
             r'$99.5\%$ coverage.'),
    label='tab:coverage-stats',
))

\begin{table*}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lcccccccccc}
\toprule
\multirow{2}{*}{Dataset} & \multirow{2}{*}{Points} & \multirow{2}{*}{Dim.} & \multirow{2}{*}{Coverage\ (\%)} & \multicolumn{4}{c}{Out-degree} & \multicolumn{3}{c}{In-degree} \\
\cmidrule(lr){5-8} \cmidrule(lr){9-11}
& & & & Mean & Median & Min & Max & Median & Min & Max \\
\midrule
\multirow{2}{*}{coco\_i2i} & \multirow{2}{*}{113,287} & \multirow{2}{*}{512} & 100.0 & 28.59 & 27.0 & 2 & 131 & 24.0 & 1 & 268 \\
 &  &  & 99.5 & 13.54 & 12.0 & 2 & 75 & 13.0 & 1 & 87 \\
\cmidrule(l){2-11}
\multirow{2}{*}{fashion\_mnist} & \multirow{2}{*}{60,000} & \multirow{2}{*}{784} & 100.0 & 13.55 & 12.0 & 1 & 108 & 12.0 & 1 & 117 \\
 &  &  & 99.5 & 6.59 & 6.0 & 1 & 70 & 6.0 & 1 & 54 \\
\cmidrule(l){2-11}
\multirow{2}{*}{glove25} & \multirow{2}{*}{1,183,514} & \multirow{2}{*}{25} & 100.0 & 50.41 & 50.0 & 1 & 141 & 43.0 & 1 & 422 \\
 &  &  & 99.5 & 8.29 & 7.0 & 1 & 71 & 7.0 & 1 & 61 \\
\cmidrule(l){2-11}
